In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
paper_sed_plotter_corrected.py
==============================

Script CORREGIDO para plotear SEDs de cúmulos globulares con valores
consistentes (bayesianos preferidos) para publicación.

Autor: Luis A. Gutiérrez Soto
Versión: 2.0 (Consistente con plotter principal)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import ScalarFormatter, LogLocator, FixedLocator
import scipy.stats as stats
import warnings
import sys
from pathlib import Path
from astropy.io import fits
from astropy.table import Table

warnings.filterwarnings('ignore')


In [ ]:
# ============================================================================
# CONFIGURACIÓN DE ESTILO
# ============================================================================

def setup_publication_style(style='aa', font_size=9):
    """Configura estilo para publicaciones."""
    
    styles = {
        'aa': {
            'font.family': 'serif',
            'font.serif': ['Times New Roman', 'Times'],
            'font.size': 9,
            'axes.titlesize': 10,
            'axes.labelsize': 9,
            'xtick.labelsize': 8,
            'ytick.labelsize': 8,
            'legend.fontsize': 8,
            'figure.titlesize': 11,
            'figure.dpi': 300,
            'savefig.dpi': 600,
            'savefig.format': 'pdf',
            'axes.linewidth': 0.8,
            'lines.linewidth': 1.0,
            'lines.markersize': 4,
            'patch.linewidth': 0.6,
            'xtick.major.width': 0.6,
            'ytick.major.width': 0.6,
            'grid.linewidth': 0.4,
            'axes.grid': False,
            'mathtext.fontset': 'stix',
        }
    }
    
    if style in styles:
        plt.rcParams.update(styles[style])
    else:
        plt.rcParams.update(styles['aa'])
    
    return {
        'blue': '#1f77b4', 'orange': '#ff7f0e', 'green': '#2ca02c',
        'red': '#d62728', 'purple': '#9467bd', 'yellow': '#bcbd22',
        'pink': '#e91e63', 'cyan': '#00bcd4'
    }

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def convert_metallicity_to_FeH(Z, Z_solar=0.0152):
    """Convertir metalicidad Z a [Fe/H]"""
    if Z <= 0:
        print(f"⚠️  Metalicidad no positiva: Z = {Z:.6f}")
        return -np.inf
    
    FeH = np.log10(Z / Z_solar)
    return FeH

def get_physical_parameters(obj_id, results, use_bayesian=True, Z_solar=0.0152):
    """
    Obtener parámetros físicos del objeto de forma robusta.
    
    Parameters:
    -----------
    obj_id : str or int
        ID del objeto
    results : Table
        Tabla de resultados de CIGALE
    use_bayesian : bool
        Si True, prefiere valores bayesianos (continuos)
    Z_solar : float
        Metalicidad solar para conversión a [Fe/H]
    
    Returns:
    --------
    dict : Diccionario con parámetros físicos
    """
    try:
        res_obj = results[results['id'] == obj_id]
        
        if len(res_obj) == 0:
            raise ValueError(f"Objeto {obj_id} no encontrado en results")
        
        # Determinar prefijo basado en preferencia
        prefix = 'bayes.' if use_bayesian else 'best.'
        
        # Lista de columnas en orden de prioridad
        age_cols = [
            f'{prefix}stellar.age_m_star',    # Primera opción
            'best.stellar.age_m_star' if use_bayesian else 'bayes.stellar.age_m_star',  # Alternativa
            'best.sfh.age_main'               # Última opción
        ]
        
        metal_cols = [
            f'{prefix}stellar.metallicity',   # Primera opción
            'best.stellar.metallicity' if use_bayesian else 'bayes.stellar.metallicity'  # Alternativa
        ]
        
        mass_cols = [
            f'{prefix}stellar.m_star',        # Primera opción
            'best.stellar.m_star' if use_bayesian else 'bayes.stellar.m_star'  # Alternativa
        ]
        
        # Función auxiliar para encontrar primera columna disponible
        def find_first_col(col_list, obj_data):
            for col in col_list:
                if col in obj_data.colnames:
                    value = obj_data[col][0]
                    if not np.isnan(value) and value is not None:
                        return col, value
            return None, None
        
        # Buscar parámetros
        age_col, age_value = find_first_col(age_cols, res_obj)
        if age_value is None:
            raise ValueError(f"No se encontró columna de edad válida para {obj_id}")
        
        metal_col, metal_value = find_first_col(metal_cols, res_obj)
        if metal_value is None:
            raise ValueError(f"No se encontró columna de metalicidad válida para {obj_id}")
        
        mass_col, mass_value = find_first_col(mass_cols, res_obj)
        if mass_value is None:
            raise ValueError(f"No se encontró columna de masa válida para {obj_id}")
        
        # Obtener chi2 reducido
        chi2_keys = ['best.reduced_chi_square', 'reduced_chi_square']
        reduced_chi2 = None
        for key in chi2_keys:
            if key in res_obj.colnames:
                reduced_chi2 = res_obj[key][0]
                break
        
        # Verificar si los valores son discretos
        def check_discreteness(col_name, values):
            """Verifica si los valores parecen discretos (del grid)"""
            if col_name is None or values is None:
                return False, 0
            
            # Tomar todos los valores de esa columna (sin NaN)
            all_vals = values[~np.isnan(values)]
            if len(all_vals) == 0:
                return False, 0
            
            unique_vals = np.unique(all_vals)
            n_unique = len(unique_vals)
            total = len(all_vals)
            
            # Si hay menos del 10% de valores únicos, probablemente es discreto
            is_discrete = n_unique / total < 0.1 if total > 0 else False
            
            return is_discrete, n_unique
        
        # Convertir a [Fe/H]
        FeH = convert_metallicity_to_FeH(metal_value, Z_solar)
        
        # Convertir edad de Myr a Gyr para display
        age_Gyr = age_value / 1000.0 if age_value > 1000 else age_value
        
        params = {
            'age': age_value,           # En Myr
            'age_Gyr': age_Gyr,         # En Gyr
            'age_col': age_col,
            'metallicity': metal_value,
            'metallicity_col': metal_col,
            'FeH': FeH,
            'mass': mass_value,
            'mass_col': mass_col,
            'reduced_chi2': reduced_chi2,
            'method': 'BAYESIAN' if 'bayes' in age_col else 'BEST',
            'use_bayesian': 'bayes' in age_col
        }
        
        return params
        
    except Exception as e:
        print(f"❌ Error en get_physical_parameters: {e}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# CLASE PRINCIPAL PARA SED
# ============================================================================

class SEDPlotter:
    """Plotter de SEDs con valores consistentes para publicación."""
    
    def __init__(self, results_file, observations_file, output_dir='sed_plots_corrected', 
                 style='aa', Z_solar=0.0152):
        """
        Inicializar plotter de SED.
        
        Parameters:
        -----------
        results_file : str
            Ruta al archivo results.fits de CIGALE
        observations_file : str
            Ruta al archivo observations.fits de CIGALE
        output_dir : str
            Directorio de salida para los gráficos
        style : str
            Estilo de matplotlib a usar
        Z_solar : float
            Metalicidad solar para conversión
        """
        self.results_file = Path(results_file)
        self.observations_file = Path(observations_file)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        
        self.colors = setup_publication_style(style)
        self.style = style
        self.Z_solar = Z_solar
        
        # Cargar datos
        self.results = self.load_results()
        self.observations = self.load_observations()
        
        print(f"✅ Cargados {len(self.results)} objetos de resultados")
        print(f"✅ Cargados {len(self.observations)} objetos de observaciones")
    
    def load_results(self):
        """Cargar archivo de resultados de CIGALE."""
        if not self.results_file.exists():
            raise FileNotFoundError(f"No se encuentra: {self.results_file}")
        
        print(f"📊 Cargando resultados desde: {self.results_file}")
        with fits.open(self.results_file) as hdul:
            results = Table(hdul[1].data)
        
        # Verificar columnas importantes
        bayes_cols = [col for col in results.colnames if 'bayes' in col]
        best_cols = [col for col in results.colnames if 'best' in col]
        
        print(f"   • Columnas bayesianas: {len(bayes_cols)}")
        print(f"   • Columnas 'best': {len(best_cols)}")
        
        return results
    
    def load_observations(self):
        """Cargar archivo de observaciones de CIGALE."""
        if not self.observations_file.exists():
            raise FileNotFoundError(f"No se encuentra: {self.observations_file}")
        
        print(f"📊 Cargando observaciones desde: {self.observations_file}")
        with fits.open(self.observations_file) as hdul:
            observations = Table(hdul[1].data)
        
        return observations
    
    def get_object_data(self, obj_id, use_bayesian=True):
        """
        Obtener todos los datos necesarios para un objeto.
        
        Returns:
        --------
        dict : Diccionario con todos los datos del objeto
        """
        # Convertir obj_id a tipo apropiado
        try:
            obj_id_str = str(obj_id)
            # Buscar objeto en resultados
            res_mask = self.results['id'] == obj_id_str
            if not np.any(res_mask):
                # Intentar como número
                try:
                    obj_id_num = int(obj_id)
                    res_mask = self.results['id'] == obj_id_num
                except:
                    pass
            
            if not np.any(res_mask):
                raise ValueError(f"Objeto {obj_id} no encontrado en resultados")
            
            res_obj = self.results[res_mask]
            obs_obj = self.observations[self.observations['id'] == obj_id_str]
            
            # Obtener parámetros físicos
            params = get_physical_parameters(obj_id_str, self.results, use_bayesian, self.Z_solar)
            
            if params is None:
                raise ValueError(f"No se pudieron obtener parámetros para {obj_id}")
            
            # Cargar modelo SED
            model_path = self.results_file.parent / f"{obj_id_str}_best_model.fits"
            if not model_path.exists():
                raise FileNotFoundError(f"No se encuentra modelo SED: {model_path}")
            
            model = Table.read(model_path)
            
            # Preparar datos de filtros
            filters_data = self.prepare_filters_data(res_obj, obs_obj)
            
            data = {
                'obj_id': obj_id_str,
                'res_obj': res_obj,
                'obs_obj': obs_obj,
                'model': model,
                'params': params,
                'filters': filters_data,
                'model_path': model_path
            }
            
            return data
            
        except Exception as e:
            print(f"❌ Error obteniendo datos para {obj_id}: {e}")
            return None
    
    def prepare_filters_data(self, res_obj, obs_obj):
        """Preparar datos de filtros para plotting."""
        
        # Definir filtros S-PLUS
        splus_filters = {
            'F0378': (0.378, '#4A148C', 'J0378'),  # Púrpura oscuro
            'F0395': (0.395, '#6A1B9A', 'J0395'),  # Púrpura  
            'F0410': (0.410, '#283593', 'J0410'),  # Azul índigo
            'F0430': (0.430, '#0277BD', 'J0430'),  # Azul
            'F0515': (0.515, '#00897B', 'J0515'),  # Verde azulado
            'F0660': (0.660, '#E65100', 'J0660'),  # Naranja oscuro
            'F0861': (0.861, '#BF360C', 'J0861')   # Rojo oscuro
        }
        
        # Filtros DECam/SDSS
        decam_filters = {
            'sdss.up': (0.354, '#7B1FA2', 'u'),   # Púrpura
            'sdss.gp': (0.477, '#1976D2', 'g'),   # Azul
            'sdss.rp': (0.623, '#D32F2F', 'r'),   # Rojo
            'sdss.ip': (0.762, '#F57C00', 'i'),   # Naranja
            'sdss.zp': (0.913, '#388E3C', 'z')    # Verde
        }
        
        filters_data = []
        
        # Procesar filtros S-PLUS
        for filter_name, (wave, color, band_short) in splus_filters.items():
            if filter_name in obs_obj.colnames and f'best.{filter_name}' in res_obj.colnames:
                flux_obs = obs_obj[filter_name][0]
                flux_err = obs_obj[filter_name + '_err'][0]
                flux_model = res_obj[f'best.{filter_name}'][0]
                
                # Calcular residuo normalizado
                if flux_err > 0:
                    residual = (flux_obs - flux_model) / flux_err
                else:
                    residual = np.nan
                
                filters_data.append({
                    'band': f'S-PLUS {band_short}',
                    'filter_name': filter_name,
                    'wave': wave,
                    'obs': flux_obs,
                    'model': flux_model,
                    'error': flux_err,
                    'residual': residual,
                    'color': color,
                    'marker': 's',  # Cuadrado para S-PLUS
                    'type': 'S-PLUS'
                })
        
        # Procesar filtros DECam/SDSS
        for filter_name, (wave, color, band_short) in decam_filters.items():
            if filter_name in obs_obj.colnames and f'best.{filter_name}' in res_obj.colnames:
                flux_obs = obs_obj[filter_name][0]
                flux_err = obs_obj[filter_name + '_err'][0]
                flux_model = res_obj[f'best.{filter_name}'][0]
                
                # Calcular residuo normalizado
                if flux_err > 0:
                    residual = (flux_obs - flux_model) / flux_err
                else:
                    residual = np.nan
                
                filters_data.append({
                    'band': f'DECam {band_short}',
                    'filter_name': filter_name,
                    'wave': wave,
                    'obs': flux_obs,
                    'model': flux_model,
                    'error': flux_err,
                    'residual': residual,
                    'color': color,
                    'marker': '^',  # Triángulo para DECam
                    'type': 'DECam'
                })
        
        return filters_data
    
    def plot_sed(self, obj_id, use_bayesian=True, show_residuals=True, 
                 figsize=(12, 10), filename=None):
        """
        Plotear SED para un objeto específico.
        
        Parameters:
        -----------
        obj_id : str or int
            ID del objeto
        use_bayesian : bool
            Usar valores bayesianos (True) o best (False)
        show_residuals : bool
            Mostrar panel de residuos
        figsize : tuple
            Tamaño de la figura
        filename : str
            Nombre del archivo de salida
        """
        print(f"\n{'='*70}")
        print(f"📊 PROCESANDO OBJETO: {obj_id}")
        print(f"{'='*70}")
        
        # Obtener datos del objeto
        data = self.get_object_data(obj_id, use_bayesian)
        if data is None:
            print(f"❌ No se pudo obtener datos para {obj_id}")
            return None
        
        # Extraer componentes
        model = data['model']
        params = data['params']
        filters_data = data['filters']
        
        # Configurar figura
        if show_residuals:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize,
                                          gridspec_kw={'height_ratios': [2, 1]})
        else:
            fig, ax1 = plt.subplots(1, 1, figsize=(figsize[0], figsize[1]*0.7))
            ax2 = None
        
        # Convertir nm a μm para mejor visualización
        wavelength_um = model['wavelength'] / 1000.0
        
        # --- Panel 1: SED completo ---
        # Modelo total
        ax1.loglog(wavelength_um, model['Fnu'], 'k-', linewidth=1.0,
                  label='Total Model', alpha=0.6)
        
        # Componentes del modelo
        if 'stellar' in model.colnames:
            ax1.loglog(wavelength_um, model['stellar'], 'steelblue', alpha=0.6,
                      linewidth=1.0, label='Stellar Component', linestyle='--')
        
        # Plotear puntos observados y del modelo
        self.plot_observed_points(ax1, filters_data)
        
        # Configurar eje Y con formato científico limpio
        self.configure_yaxis(ax1)
        
        # Configurar título y etiquetas
        self.configure_axes_and_title(ax1, data, show_residuals)
        
        # Leyenda
        self.add_legend(ax1, filters_data)
        
        # Grid
        ax1.grid(True, alpha=0.2, linestyle='-', linewidth=0.5)
        ax1.set_xlim(0.3, 1.0)
        
        if show_residuals:
            ax1.xaxis.set_visible(False)
        
        # --- Panel 2: Residuos ---
        if show_residuals and filters_data:
            self.plot_residuals(ax2, filters_data)
        
        # Ajustar layout
        plt.tight_layout()
        if show_residuals:
            plt.subplots_adjust(hspace=0.05)
        
        # Guardar figura
        if filename is None:
            method = 'bayes' if use_bayesian else 'best'
            filename = f"sed_{obj_id}_{method}.pdf"
        
        output_path = self.output_dir / filename
        plt.savefig(output_path, dpi=600, bbox_inches='tight')
        print(f"✅ SED guardado en: {output_path}")
        
        # Guardar también como PNG para revisión rápida
        png_path = output_path.with_suffix('.png')
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        
        plt.show()
        
        # Imprimir resumen estadístico
        if filters_data:
            self.print_statistics(obj_id, filters_data, params)
        
        return fig, (ax1, ax2)
    
    def plot_observed_points(self, ax, filters_data):
        """Plotear puntos observados y del modelo."""
        # Plotear puntos observados
        for data in filters_data:
            if data['type'] == 'S-PLUS':
                marker = 's'  # Cuadrado
            else:
                marker = '^'  # Triángulo
            
            # Punto observado
            ax.errorbar(data['wave'], data['obs'], yerr=data['error'],
                       fmt=marker, markerfacecolor='none', markeredgecolor=data['color'],
                       markersize=10, capsize=3, markeredgewidth=1.5,
                       ecolor=data['color'], alpha=0.9, zorder=5,
                       label=data['band'])
        
        # Plotear puntos del modelo
        model_plotted = False
        for data in filters_data:
            if not model_plotted:
                ax.plot(data['wave'], data['model'], marker='o', markersize=6,
                       markerfacecolor=self.colors['pink'], markeredgecolor=self.colors['pink'],
                       linestyle='', label='Model fluxes', alpha=0.9, zorder=4)
                model_plotted = True
            else:
                ax.plot(data['wave'], data['model'], marker='o', markersize=6,
                       markerfacecolor=self.colors['pink'], markeredgecolor=self.colors['pink'],
                       linestyle='', alpha=0.9, zorder=4)
    
    def configure_yaxis(self, ax):
        """Configurar eje Y con formato científico limpio."""
        class CleanScientificFormatter(ScalarFormatter):
            def __init__(self):
                ScalarFormatter.__init__(self)
                self.set_scientific(True)
                self.set_powerlimits((-2, 2))
            
            def format_data(self, value):
                if value == 0:
                    return "0"
                exponent = np.floor(np.log10(abs(value)))
                coeff = value / 10**exponent
                coeff_rounded = round(coeff, 1)
                if coeff_rounded == 10.0:
                    coeff_rounded = 1.0
                    exponent += 1
                return f"${coeff_rounded:.1f} \\times 10^{{{int(exponent)}}}$"
        
        ax.yaxis.set_major_formatter(CleanScientificFormatter())
        ax.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0, 2.0, 5.0), numticks=12))
        
        # Ajustar rango Y automáticamente
        all_fluxes = []
        for data in self.filters_data if hasattr(self, 'filters_data') else []:
            all_fluxes.extend([data['obs'], data['model']])
        
        if all_fluxes:
            min_flux = min(all_fluxes) * 0.3
            max_flux = max(all_fluxes) * 3.0
            min_exp = np.floor(np.log10(min_flux))
            max_exp = np.ceil(np.log10(max_flux))
            ax.set_ylim(10**min_exp, 10**max_exp)
    
    def configure_axes_and_title(self, ax, data, show_residuals):
        """Configurar ejes y título."""
        obj_id = data['obj_id']
        params = data['params']
        
        if show_residuals:
            ax.set_ylabel('Flux (mJy)', fontsize=14, fontweight='normal')
        else:
            ax.set_xlabel('Wavelength (μm)', fontsize=14, fontweight='normal')
            ax.set_ylabel('Flux (mJy)', fontsize=14, fontweight='normal')
        
        # Título con parámetros físicos
        title_text = (f'GC {obj_id} | '
                     f'M = {params["mass"]:.2e} M$_\odot$ | '
                     f'[Fe/H] = {params["FeH"]:.2f} | '
                     f'Age = {params["age_Gyr"]:.1f} Gyr')
        
        if params['reduced_chi2'] is not None:
            title_text += f' | $\chi^2$ = {params["reduced_chi2"]:.2f}'
        
        title_text += f' | ({params["method"]})'
        
        ax.set_title(title_text, fontsize=14, pad=10)
        
        # Añadir nota sobre método
        method_note = f'Method: {params["method"]}\n'
        method_note += f'Age col: {params["age_col"]}\n'
        method_note += f'Metal col: {params["metallicity_col"]}'
        
        ax.text(0.02, 0.98, method_note, transform=ax.transAxes,
               fontsize=8, verticalalignment='top',
               style='italic', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))
    
    def add_legend(self, ax, filters_data):
        """Añadir leyenda organizada."""
        handles, labels = ax.get_legend_handles_labels()
        
        # Separar por tipo
        model_handles = [h for h, l in zip(handles, labels) if 'Model' in l or 'Component' in l]
        model_labels = [l for l in labels if 'Model' in l or 'Component' in l]
        
        splus_handles = [h for h, l in zip(handles, labels) if 'S-PLUS' in l]
        splus_labels = [l for l in labels if 'S-PLUS' in l]
        
        decam_handles = [h for h, l in zip(handles, labels) if 'DECam' in l]
        decam_labels = [l for l in labels if 'DECam' in l]
        
        # Reconstruir ordenada
        ordered_handles = model_handles + splus_handles + decam_handles
        ordered_labels = model_labels + splus_labels + decam_labels
        
        # Crear leyenda
        n_columns = 3 if len(ordered_labels) > 10 else 2
        ax.legend(ordered_handles, ordered_labels, loc='best',
                 ncol=n_columns, fontsize=9, framealpha=0.9, fancybox=True,
                 columnspacing=0.8, handletextpad=0.5)
    
    def plot_residuals(self, ax, filters_data):
        """Plotear residuos."""
        # Ordenar por longitud de onda
        filters_data.sort(key=lambda x: x['wave'])
        
        bands = [d['band'] for d in filters_data]
        waves = [d['wave'] for d in filters_data]
        residuals = [d['residual'] for d in filters_data]
        colors = [d['color'] for d in filters_data]
        markers = [d['marker'] for d in filters_data]
        
        # Plot residuos
        for i, (band, wave, residual, color, marker) in enumerate(zip(bands, waves, residuals, colors, markers)):
            if not np.isnan(residual):
                ax.scatter(wave, residual, color=color, s=80, zorder=3,
                          marker=marker, alpha=0.8, edgecolors='black', linewidth=0.5)
                
                # Conectar puntos
                if i > 0 and not np.isnan(residuals[i-1]):
                    prev_wave = waves[i-1]
                    prev_residual = residuals[i-1]
                    ax.plot([prev_wave, wave], [prev_residual, residual],
                           'gray', alpha=0.3, linestyle='-', linewidth=1)
        
        # Líneas de referencia
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.8, linewidth=1)
        ax.axhline(y=1, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
        ax.axhline(y=-1, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
        ax.axhline(y=2, color='red', linestyle=':', alpha=0.3, linewidth=0.6)
        ax.axhline(y=-2, color='red', linestyle=':', alpha=0.3, linewidth=0.6)
        
        ax.set_xlabel('Wavelength (μm)', fontsize=14, fontweight='normal')
        ax.set_ylabel('Residual (σ)', fontsize=14, fontweight='normal')
        
        ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5)
        ax.set_xscale('log')
        ax.set_xlim(0.3, 1.0)
        
        # Formatear eje X
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.xaxis.set_major_locator(FixedLocator([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]))
        ax.set_xticks([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
        ax.set_xticklabels(['0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9', '1.0'])
        
        # Ajustar límites Y
        valid_residuals = [r for r in residuals if not np.isnan(r)]
        if valid_residuals:
            max_res = max(3, max(np.abs(valid_residuals)) * 1.2)
            ax.set_ylim(-max_res, max_res)
    
    def print_statistics(self, obj_id, filters_data, params):
        """Imprimir estadísticas detalladas."""
        print(f"\n{'='*70}")
        print(f"ANÁLISIS DE RESIDUOS - Cúmulo Globular {obj_id}")
        print(f"{'='*70}")
        print(f"{'Filtro':<15} {'λ (μm)':<8} {'Obs (mJy)':<12} {'Model (mJy)':<12} {'Residuo (σ)':<12} {'|Residuo|':<10}")
        print(f"{'-'*70}")
        
        for data in filters_data:
            abs_residual = abs(data['residual']) if not np.isnan(data['residual']) else np.nan
            residual_str = f"{data['residual']:.2f}" if not np.isnan(data['residual']) else "NaN"
            abs_str = f"{abs_residual:.2f}" if not np.isnan(abs_residual) else "NaN"
            
            print(f"{data['band']:<15} {data['wave']:<8.3f} {data['obs']:<12.6f} "
                  f"{data['model']:<12.6f} {residual_str:<12} {abs_str:<10}")
        
        print(f"{'-'*70}")
        
        # Calcular estadísticas
        valid_residuals = [d['residual'] for d in filters_data if not np.isnan(d['residual'])]
        if valid_residuals:
            abs_residuals = [abs(r) for r in valid_residuals]
            avg_abs_residual = np.mean(abs_residuals)
            
            print(f"Residuo absoluto promedio: {avg_abs_residual:.2f}σ")
            print(f"Número de puntos válidos: {len(valid_residuals)}/{len(filters_data)}")
            
            # Separar por tipo de filtro
            splus_residuals = [abs(d['residual']) for d in filters_data 
                              if d['type'] == 'S-PLUS' and not np.isnan(d['residual'])]
            decam_residuals = [abs(d['residual']) for d in filters_data 
                              if d['type'] == 'DECam' and not np.isnan(d['residual'])]
            
            if splus_residuals:
                print(f"S-PLUS (estrechos): {np.mean(splus_residuals):.2f}σ promedio ({len(splus_residuals)} filtros)")
            if decam_residuals:
                print(f"DECam/SDSS (anchos): {np.mean(decam_residuals):.2f}σ promedio ({len(decam_residuals)} filtros)")
        
        # Información de parámetros
        print(f"\n📊 PARÁMETROS FÍSICOS:")
        print(f"   • Edad: {params['age']:.1f} Myr ({params['age_Gyr']:.2f} Gyr)")
        print(f"   • Fuente edad: {params['age_col']}")
        print(f"   • Metalicidad Z: {params['metallicity']:.6f}")
        print(f"   • [Fe/H]: {params['FeH']:.3f}")
        print(f"   • Fuente metalicidad: {params['metallicity_col']}")
        print(f"   • Masa estelar: {params['mass']:.2e} M_sun")
        print(f"   • Método: {params['method']}")
        if params['reduced_chi2'] is not None:
            print(f"   • χ² reducido: {params['reduced_chi2']:.2f}")
        
        print(f"{'='*70}")
    
    def plot_multiple_objects(self, obj_ids, use_bayesian=True, **kwargs):
        """
        Plotear SEDs para múltiples objetos.
        
        Parameters:
        -----------
        obj_ids : list
            Lista de IDs de objetos
        use_bayesian : bool
            Usar valores bayesianos
        **kwargs : 
            Argumentos adicionales para plot_sed
        """
        for obj_id in obj_ids:
            try:
                self.plot_sed(obj_id, use_bayesian=use_bayesian, **kwargs)
            except Exception as e:
                print(f"❌ Error procesando {obj_id}: {e}")
                import traceback
                traceback.print_exc()

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

def main():
    """Función principal."""
    
    print("\n" + "="*70)
    print("📊 SED PLOTTER CORREGIDO - Valores Bayesianos Consistentes")
    print("="*70)
    
    # Resolver rutas desde la raíz del repositorio o desde Notebooks/.
    working_dir = Path.cwd().resolve()
    repo_root = working_dir if (working_dir / 'anac_data').is_dir() else working_dir.parent
    corrected_catalog = repo_root / 'anac_data/Results_Corrected/all_fields_photometry_OPTIMIZED_1394objects.csv'
    cigale_dir = repo_root / 'anac_data/CIGALE-DEcam-extinction'
    results_file = cigale_dir / 'out/results.fits'
    observations_file = cigale_dir / 'out/observations.fits'
    output_dir = cigale_dir / 'sed_plots_corrected'

    if corrected_catalog.is_file():
        catalog = pd.read_csv(corrected_catalog)
        if 'T17ID' not in catalog.columns:
            raise ValueError(f"No existe la columna T17ID en {corrected_catalog}")
        catalog_ids = set(catalog['T17ID'].dropna().astype(str))
        print(f'Catálogo corregido: {corrected_catalog} ({len(catalog)} objetos)')
    else:
        catalog_ids = None
        print(f'⚠️ No se encontró el catálogo corregido: {corrected_catalog}')

    if not results_file.is_file() or not observations_file.is_file():
        raise FileNotFoundError(
            f"No se encuentran resultados CIGALE en {cigale_dir / 'out'}. "
            'Ejecuta CIGALE con la configuración de CIGALE-DEcam-extinction.'
        )
    
    # Crear plotter
    plotter = SEDPlotter(
        results_file=results_file,
        observations_file=observations_file,
        output_dir=output_dir,
        style='aa',
        Z_solar=0.0152
    )
    
    # Lista de objetos a plotear (ajustar con tus IDs)
    obj_ids = [
        'T17-2602', 'T17-2211', 'T17-2684'  # Ejemplo, reemplazar con tus IDs reales
    ]
    if catalog_ids is not None:
        missing_ids = [obj_id for obj_id in obj_ids if obj_id not in catalog_ids]
        if missing_ids:
            print(f'⚠️ IDs ausentes del catálogo corregido: {missing_ids}')
    
    # Plotear todos los objetos
    plotter.plot_multiple_objects(
        obj_ids=obj_ids,
        use_bayesian=True,  # ¡IMPORTANTE! Usar valores bayesianos para consistencia
        show_residuals=True,
        figsize=(12, 10)
    )
    
    print(f"\n✅ ANÁLISIS COMPLETADO!")
    print(f"📁 Resultados en: {output_dir}")

if __name__ == "__main__":
    main()